In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def endpoint_target_prob(dist: torch.Tensor, beta: float) -> float:
    dist_f = dist.float()
    log_p = -beta * dist_f
    log_p = log_p - log_p.logsumexp(dim=-1, keepdim=True)
    p_target = log_p.diagonal(dim1=-2, dim2=-1).exp()  
    return p_target.min().item()


@torch.no_grad()
def find_beta_max(
    dist: torch.Tensor,
    target_prob: float = 1 - 1e-8,
    beta_init_hi: float = 1.0,
    max_binary_steps: int = 100,
):

    lo = 0.0
    hi = beta_init_hi

    val = endpoint_target_prob(dist, hi)
    n_expand = 0
    while val < target_prob:
        hi *= 2.0
        val = endpoint_target_prob(dist, hi)
        n_expand += 1

    for _ in range(max_binary_steps):
        mid = 0.5 * (lo + hi)
        val = endpoint_target_prob(dist, mid)
        if val >= target_prob:
            hi = mid
        else:
            lo = mid

    return hi

@torch.no_grad()
def precompute_beta_from_fisher(
    dist: torch.Tensor,
    beta_grid_max: float,
    n_points: int = 1024,
    beta_grid_size: int = 4096,
    eps: float = 1e-8,
):

    if dist.dim() == 2:
        dist = dist.unsqueeze(0)
    c = dist.shape[0]
    device = dist.device
    dtype = dist.dtype

    beta_grid = torch.linspace(0.0, beta_grid_max, beta_grid_size, device=device, dtype=torch.float32)

    V_grid = torch.empty(c, beta_grid_size, device=device, dtype=torch.float32)

    for k, beta in enumerate(beta_grid):
        log_p = - beta * dist.float()                             
        log_p = log_p - log_p.logsumexp(dim=-1, keepdim=True)
        p = log_p.exp()

        mean_d = (p * dist).sum(dim=-1)                           
        mean_d2 = (p * (dist ** 2)).sum(dim=-1)                  
        var_d = (mean_d2 - mean_d ** 2).clamp_min(0.0)             

        V_grid[:, k] = var_d.mean(dim=-1)                         

    V_use = V_grid.mean(dim=0, keepdim=True)                  

    sqrtV = torch.sqrt(V_use.clamp_min(eps))                       
    d_beta = beta_grid[1:] - beta_grid[:-1]                        
    trap = 0.5 * (sqrtV[:, 1:] + sqrtV[:, :-1]) * d_beta.unsqueeze(0)  

    s = torch.zeros(V_use.shape[0], beta_grid_size, device=device, dtype=torch.float32)
    s[:, 1:] = torch.cumsum(trap, dim=-1)                         
    s_total = s[:, -1].clamp_min(eps)                              

    ts = torch.linspace(0.0, 1.0, n_points, device=device, dtype=torch.float32)
    s_target = s_total[:, None] * ts[None, :]                     

    beta = torch.empty_like(s_target)
    for m in range(s.shape[0]):
        idx = torch.searchsorted(s[m], s_target[m], right=False)   
        idx = idx.clamp(min=1, max=beta_grid_size - 1)

        s0 = s[m, idx - 1]
        s1 = s[m, idx]
        b0 = beta_grid[idx - 1]
        b1 = beta_grid[idx]

        w = (s_target[m] - s0) / (s1 - s0 + eps)
        beta[m] = b0 + w * (b1 - b0)


    V0 = V_use[0]  
    idx = torch.searchsorted(beta_grid, beta[0], right=False)  
    idx = idx.clamp(min=1, max=beta_grid_size - 1)

    b0 = beta_grid[idx - 1]
    b1 = beta_grid[idx]
    v0 = V0[idx - 1]
    v1 = V0[idx]

    w = (beta[0] - b0) / (b1 - b0 + eps)
    V_at_beta = v0 + w * (v1 - v0)                              

    beta_dt = (s_total[0] / torch.sqrt(V_at_beta.clamp_min(eps))).unsqueeze(0) 

    beta = beta[0].to(dtype)
    beta_dt = beta_dt[0].to(dtype)

    return ts, beta, beta_dt

In [13]:
embeds = torch.load("./pretrained/codebook_embeds.pt") # [12, 1024, 8]
embeds = F.normalize(embeds, dim=-1) # codec uses L2 norm

dist = torch.cdist(embeds, embeds, p=2) ** 2  # distance matrix
dist.diagonal(dim1=-2, dim2=-1).zero_()

beta_max = find_beta_max(
    dist,
    target_prob=1 - 1e-8,
)

ts, beta, beta_dt = precompute_beta_from_fisher(
        dist,
        beta_grid_max=beta_max,
        n_points=1024,
        beta_grid_size=4096,
    )

torch.save(beta, "beta.pt")
torch.save(beta_dt, "beta_dt.pt")